# Multi-modal U-Net Patient-level Predictor & Interactive Visualizer

This notebook allows you to select a trained U-Net model checkpoint (`.h5` file) and a specific test patient to:
1. Predict the full 3D lesion segmentation mask slice-by-slice.
2. Symmetrically crop or pad the slices back to their original input shape.
3. Save the predicted mask as a NIfTI volume (`pred_mask.nii.gz`) inside the results folder.
4. Save the final patient-level metrics (Dice, IoU, confusion matrix) as `prediction_details.json` inside the results folder.
5. Interactively visualize all modalities (T1, T2, FLAIR), the Ground Truth mask, and the Predicted mask slice-by-slice with a single slider.

In [3]:
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
import sys
import json
import numpy as np
import nibabel as nib
import tensorflow as tf
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import segmentation_models as sm
import sys

# Ensure path imports from src
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from dataset import centre_pad_crop_2d, load_volume, extract_slices
from model import Model2D

2026-06-04 09:46:24.348752: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-04 09:46:25.039923: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Segmentation Models: using `tf.keras` framework.


### 1. Configuration
Specify the trained model checkpoint path and the patient ID in the test dataset.

In [4]:
# Path to the model checkpoint and test subject
model_path = "/home/darshan/MS/models/unet/runs/adam_1e-05_8_1/best_model.h5"
subject_id = "S21"  # Choose a subject from the test split (e.g. S7, S47, S152, S149, S12, S105)

### 2. Run Predictions & Save NIfTI + JSON Results

In [5]:
# Verify paths
script_dir = os.getcwd()
preprocessed_dir = os.path.abspath(os.path.join(script_dir, "..", "..", "..", "data", "PREPROCESSED"))
subj_dir = os.path.join(preprocessed_dir, subject_id)

t1_path = os.path.join(subj_dir, "t1.nii.gz")
t2_path = os.path.join(subj_dir, "t2.nii.gz")
flair_path = os.path.join(subj_dir, "flair.nii.gz")
mask_path = os.path.join(subj_dir, "mask.nii.gz")

if not all(os.path.exists(p) for p in [t1_path, t2_path, flair_path, mask_path]):
    raise FileNotFoundError(f"One or more NIfTI files missing for subject {subject_id} in {subj_dir}")

# Extract model run name to determine results folder
run_name = os.path.basename(os.path.dirname(model_path))
results_dir = os.path.abspath(os.path.join(script_dir, "..", "results", run_name))
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")

# 1. Load original volume data (unpadded)
t1_img = nib.load(t1_path)
t1_data = t1_img.get_fdata(dtype=np.float32)
t2_data = nib.load(t2_path).get_fdata(dtype=np.float32)
flair_data = nib.load(flair_path).get_fdata(dtype=np.float32)
mask_data = (nib.load(mask_path).get_fdata(dtype=np.float32) > 0.5).astype(np.float32)

H_orig, W_orig, D_orig = flair_data.shape
print(f"Original volume dimensions: Height={H_orig}, Width={W_orig}, Depth={D_orig}")

# 2. Load model
print("Loading trained U-Net model checkpoint...")
class DummyDiceScore(tf.keras.metrics.Metric):
    def __init__(self, **kwargs): super().__init__(name='dice_score')
class DummyIoUScore(tf.keras.metrics.Metric):
    def __init__(self, **kwargs): super().__init__(name='iou_score')

model = tf.keras.models.load_model(
    model_path,
    custom_objects={
        'dice_loss': sm.losses.dice_loss,
        'dice_score': DummyDiceScore,
        'iou_score': DummyIoUScore
    },
    compile=False
)

# 3. Extract prepared slices for prediction
print("Preparing modalities for model prediction...")
imgs_prep, masks_prep = extract_slices(t1_path, t2_path, flair_path, mask_path, is_train=False, skip_blank_ratio=0.0)

# Predict slice-by-slice
print("Running model inference...")
preds = model.predict(imgs_prep, batch_size=4, verbose=1)
preds_bin = (preds > 0.5).astype(np.float32)

# 4. Reconstruct 3D Predicted Mask (pad/crop back to original size)
pred_mask_3d = np.zeros_like(mask_data)
for z in range(D_orig):
    pred_slice_2d = preds_bin[z, :, :, 0]
    # Pad/crop from (182, 218) back to original shape
    pred_slice_orig = centre_pad_crop_2d(pred_slice_2d, H_orig, W_orig)
    pred_mask_3d[:, :, z] = pred_slice_orig

# 5. Save Predicted Mask as NIfTI
pred_nii = nib.Nifti1Image(pred_mask_3d, t1_img.affine, t1_img.header)
pred_mask_path = os.path.join(results_dir, "pred_mask.nii.gz")
nib.save(pred_nii, pred_mask_path)
print(f"Saved predicted mask volume: {pred_mask_path}")

# 6. Compute metrics
inter = np.sum((mask_data == 1.0) & (pred_mask_3d == 1.0))
union_dice = np.sum(mask_data == 1.0) + np.sum(pred_mask_3d == 1.0)
union_iou = np.sum((mask_data == 1.0) | (pred_mask_3d == 1.0))

dice = float((2.0 * inter) / (union_dice + 1e-7)) if union_dice > 0 else 1.0
iou = float(inter / (union_iou + 1e-7)) if union_iou > 0 else 1.0

tp = int(np.sum((mask_data == 1.0) & (pred_mask_3d == 1.0)))
tn = int(np.sum((mask_data == 0.0) & (pred_mask_3d == 0.0)))
fp = int(np.sum((mask_data == 0.0) & (pred_mask_3d == 1.0)))
fn = int(np.sum((mask_data == 1.0) & (pred_mask_3d == 0.0)))

metrics = {
    "subject_id": subject_id,
    "dice": round(dice, 4),
    "iou": round(iou, 4),
    "confusion_matrix": {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    }
}

# Save prediction details JSON
details_path = os.path.join(results_dir, "prediction_details.json")
with open(details_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved final metrics: {details_path}")
print(json.dumps(metrics, indent=2))

Results will be saved to: /home/darshan/MS/models/unet/results/adam_1e-05_8_1
Original volume dimensions: Height=182, Width=218, Depth=182
Loading trained U-Net model checkpoint...


2026-06-04 09:46:32.165009: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-04 09:46:32.400519: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-04 09:46:32.401015: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Preparing modalities for model prediction...
Running model inference...


2026-06-04 09:46:34.230182: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 86652384 exceeds 10% of free system memory.
2026-06-04 09:46:34.656130: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600


46/46 [==============================] - 4s 38ms/step
Saved predicted mask volume: /home/darshan/MS/models/unet/results/adam_1e-05_8_1/pred_mask.nii.gz
Saved final metrics: /home/darshan/MS/models/unet/results/adam_1e-05_8_1/prediction_details.json
{
  "subject_id": "S21",
  "dice": 0.7988,
  "iou": 0.665,
  "confusion_matrix": {
    "TP": 15491,
    "TN": 7197738,
    "FP": 2620,
    "FN": 5183
  }
}


### 3. Interactive Visualization
Use the slice slider below to dynamically scroll through the entire 3D volume. You will see T1, T2, FLAIR, Ground Truth, and Predicted Mask perfectly aligned!

In [ ]:
import os
import numpy as np
import ants
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output

# --- 1. CONFIGURATION ---
current_dataset = "ISBI"  # Change this to MICCAI or MSLegSeg as needed

index = {
    "ISBI": {
        "FLAIR": "/home/darshan/MS/data/PREPROCESSED/S21/flair.nii.gz",
        "T1": "/home/darshan/MS/data/PREPROCESSED/S21/t1.nii.gz",
        "PRED_MASK": "/home/darshan/MS/models/unet/results/adam_1e-05_8_1/pred_mask.nii.gz",
        "MASK": "/home/darshan/MS/data/PREPROCESSED/S21/mask.nii.gz"
    },
    # ... (Add your other datasets here if needed)
}

def load_volume_data(dataset_name):
    """Loads all 4 modalities for the dataset into numpy arrays."""
    paths = index.get(dataset_name)
    if not paths:
        return None, 0
    
    data_store = {}
    z_dims = []
    
    print(f"Loading {dataset_name} volumes...")
    for mod, path in paths.items():
        if os.path.exists(path):
            # Load with ANTs and convert to numpy immediately
            img = ants.image_read(path)
            data_store[mod] = img.numpy()
            z_dims.append(img.shape[2])
        else:
            data_store[mod] = None
    
    # Calculate safe slice range
    min_z = min(z_dims) if z_dims else 0
    return data_store, min_z

# --- 2. CREATE ANIMATION ---
def create_html_viewer(dataset_name, step=2):
    """
    Generates a portable HTML player.
    step=2: Skips every other slice to make generation faster. 
    """
    volumes, max_z = load_volume_data(dataset_name)
    if max_z == 0:
        print("Error: No data found.")
        return

    # Set up the static 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    fig.suptitle(f"{dataset_name} Interactive View", fontsize=14, fontweight='bold')
    
    # Flatten axes for easy iteration
    ax_list = {
        "FLAIR": axes[0,0], "T1": axes[0,1],
        "PRED_MASK": axes[1,0],    "MASK": axes[1,1]
    }
    
    # Initialize plots with empty data
    img_objs = {}
    for mod, ax in ax_list.items():
        vol = volumes[mod]
        if vol is not None:
            # Display middle slice initially
            im = ax.imshow(vol[:, :, max_z//2].T, cmap='gray', origin='lower', animated=True)
            ax.set_title(mod)
            ax.axis('off')
            img_objs[mod] = im
        else:
            ax.text(0.5, 0.5, "Missing", ha='center')
            ax.axis('off')

    plt.close(fig) # Prevent static plot from showing up separately

    # Update function for the animation loop
    def update(frame_idx):
        for mod, im in img_objs.items():
            # Update the image data for the new slice
            im.set_data(volumes[mod][:, :, frame_idx].T)
            # Optional: Update title with slice number (can be slow, maybe skip)
            # ax_list[mod].set_title(f"{mod} (z={frame_idx})")
        return list(img_objs.values())

    # Generate frames (using 'step' to reduce wait time)
    frames = range(0, max_z, step)
    print(f"Generating animation for {len(frames)} slices. Please wait...")
    
    anim = FuncAnimation(fig, update, frames=frames, interval=100, blit=True)
    
    # Render to JavaScript HTML
    return HTML(anim.to_jshtml())

# --- 3. RUN ---
# This will take 10-20 seconds to render, but will be perfectly smooth afterwards
html_view = create_html_viewer(current_dataset, step=2)
display(html_view)


interactive(children=(IntSlider(value=91, description='Slice', max=181), Output()), _dom_classes=('widget-inte…

<function __main__.plot_slice(z)>

USE THIS CODE

In [ ]:
import os
import numpy as np
import ants
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output

# --- 1. CONFIGURATION ---
current_dataset = "ISBI"  # Change this to MICCAI or MSLegSeg as needed

index = {
    "ISBI": {
        "FLAIR": "/home/darshan/MS/data/PREPROCESSED/S170/flair.nii.gz",
        "T1": "/home/darshan/MS/data/PREPROCESSED/S170/t1.nii.gz",
        "PRED_MASK": "/home/darshan/MS/models/unet/results/adam_1e-05_4_4/pred_mask.nii.gz",
        "MASK": "/home/darshan/MS/data/PREPROCESSED/S170/mask.nii.gz"
    },
    # ... (Add your other datasets here if needed)
}

def load_volume_data(dataset_name):
    """Loads all 4 modalities for the dataset into numpy arrays."""
    paths = index.get(dataset_name)
    if not paths:
        return None, 0
    
    data_store = {}
    z_dims = []
    
    print(f"Loading {dataset_name} volumes...")
    for mod, path in paths.items():
        if os.path.exists(path):
            # Load with ANTs and convert to numpy immediately
            img = ants.image_read(path)
            data_store[mod] = img.numpy()
            z_dims.append(img.shape[2])
        else:
            data_store[mod] = None
    
    # Calculate safe slice range
    min_z = min(z_dims) if z_dims else 0
    return data_store, min_z

# --- 2. CREATE ANIMATION ---
def create_html_viewer(dataset_name, step=2):
    """
    Generates a portable HTML player.
    step=2: Skips every other slice to make generation faster. 
    """
    volumes, max_z = load_volume_data(dataset_name)
    if max_z == 0:
        print("Error: No data found.")
        return

    # Set up the static 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    fig.suptitle(f"{dataset_name} Interactive View", fontsize=14, fontweight='bold')
    
    # Flatten axes for easy iteration
    ax_list = {
        "FLAIR": axes[0,0], "T1": axes[0,1],
        "PRED_MASK": axes[1,0],    "MASK": axes[1,1]
    }
    
    # Initialize plots with empty data
    img_objs = {}
    for mod, ax in ax_list.items():
        vol = volumes[mod]
        if vol is not None:
            # Display middle slice initially
            im = ax.imshow(vol[:, :, max_z//2].T, cmap='gray', origin='lower', animated=True)
            ax.set_title(mod)
            ax.axis('off')
            img_objs[mod] = im
        else:
            ax.text(0.5, 0.5, "Missing", ha='center')
            ax.axis('off')

    plt.close(fig) # Prevent static plot from showing up separately

    # Update function for the animation loop
    def update(frame_idx):
        for mod, im in img_objs.items():
            # Update the image data for the new slice
            im.set_data(volumes[mod][:, :, frame_idx].T)
            # Optional: Update title with slice number (can be slow, maybe skip)
            # ax_list[mod].set_title(f"{mod} (z={frame_idx})")
        return list(img_objs.values())

    # Generate frames (using 'step' to reduce wait time)
    frames = range(0, max_z, step)
    print(f"Generating animation for {len(frames)} slices. Please wait...")
    
    anim = FuncAnimation(fig, update, frames=frames, interval=100, blit=True)
    
    # Render to JavaScript HTML
    return HTML(anim.to_jshtml())

# --- 3. RUN ---
# This will take 10-20 seconds to render, but will be perfectly smooth afterwards
html_view = create_html_viewer(current_dataset, step=2)
display(html_view)
